# Network design, version 1: risk-based QKD placement

*Builds on `3b_network_design.ipynb` (version 0). Same Chattanooga-area grid, same physical limits.*

**What changed from version 0**
- **Value is measured as risk reduced, not voltage tier.** A substation is worth: MW at stake × chance its traffic is tapped (from the length of its route) × chance the BB84 alarm catches the tap (from `3a_bb84_grid_link.ipynb`) × (1 − q) for each trusted node its key passes through. The link between control centers carries data about the whole system, so it counts as the whole system's load (`cc_weight`).
- **An exact optimizer chooses the links.** This is an integer program, solved with SciPy's HiGHS solver. It finds the best total value within budget, so more budget can never mean less protection, which version 0 violated twice.
- **Cost is in dollars:** a QKD pair per link plus hardening per trusted node.
- **Every method is scored the same way:** the optimizer, a greedy method (best value per dollar), the version 0 rule, busiest links first, and random links.
- **Robustness (SMAA):** uncertain values are given ranges, not single guesses. The optimizer runs 400 times with values drawn from those ranges, and the notebook reports how often each link is chosen.

All values and ranges are in `network_design/v1_model.py` (`DEFAULTS`, `RANGES`).

In [ ]:
import importlib.util, subprocess, sys, json, os
for pkg in ("networkx", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip install -q {pkg}
from IPython.display import Image, display
D = "network_design"
for f in ("v1_model.py", "v1_run.py", "chatt_raw.json"):
    if not os.path.exists(os.path.join(D, f)):
        raise FileNotFoundError(f"Missing {D}/{f}: put the network_design/ folder next to this notebook.")
sys.path.insert(0, D)
import v1_model
print("Defaults:", v1_model.DEFAULTS)
print("Ranges tested:", v1_model.RANGES)

In [ ]:
# Run version 1 (about 1 minute): budget sweep for every method, the design at $1.2M, and 400 SMAA draws
subprocess.run([sys.executable, f"{D}/v1_run.py", f"{D}/chatt_raw.json", D, "1200000"], check=True, capture_output=True)
s = json.load(open(f"{D}/v1_summary.json"))
d = s["design"]
print(f"Design at ${s['design_budget']:,}: {len(d['links'])} QKD links, cost ${d['cost']:,}")
print(f"  risk reduced: {d['value']} = {100*d['share_of_max_value']:.0f}% of the most achievable ({s['max_value']})")
print(f"  load behind full-QKD paths: {d['mw_behind_qkd']} MW;  trusted nodes: {', '.join(d['trusted_nodes'])}")
print(f"  control centers joined by QKD: {d['control_centers_linked']}")
for sub, path in d["protected"].items(): print(f"    {sub}: {path}")
print(f"\nLargest shortfall vs. optimal at any budget: greedy {100*s['max_gap_vs_optimal']['greedy']:.1f}%, "
      f"version 0 rule {100*s['max_gap_vs_optimal']['v0_rule']:.0f}%")

In [ ]:
display(Image(f"{D}/v1_budget_curves.png"))

In [ ]:
display(Image(f"{D}/v1_smaa.png"))
print("Share of 400 parameter draws in which each link gets QKD:")
for k, v in s["smaa_link_frequency"].items(): print(f"  {100*v:5.1f}%  {k}")

In [ ]:
display(Image(f"{D}/v1_network_map.png"))

## What to look for
- **Budget curves:** the optimizer never drops as the budget grows. The jump near $0.6M is the three-hop chain joining the control centers. After that, returns shrink.
- **Greedy vs. optimal:** if greedy stays within about 1%, the network is simple enough that a careful engineer gets close. The optimizer's job is then to *guarantee* that and to rule out failures like version 0's.
- **SMAA:** links chosen in nearly every draw don't depend on the uncertain values. Those are the safe first purchases.